# 04 – Aufbereitung (Preprocessing)

Die Palepu-Variablen aus Notebook 03 werden modellfertig gemacht, nach den Entscheidungen vom 28.10.2026 (siehe `PROGRESS.md`, *Decisions*):

1. **Ausreißer:** stetige Variablen auf das 1. und 99. Perzentil kappen (winsorisieren).
2. **Fehlende Werte:** mit dem Median auffüllen und einen 0/1-Indikator `<variable>_missing` anlegen; `grdummy` wird bei fehlender Zutat 0 gesetzt, mit `grdummy_missing = 1`. Ein Indikator wird nur behalten, wenn im Training **mehr als 2 % fehlen und mindestens 5 Positive in der Lücke liegen** (Regel aus dem alten Code) – sonst kann das Logit für ihn keinen stabilen Koeffizienten schätzen.
3. **Kein Datenleck:** Grenzen und Mediane kommen **nur aus dem Training (FY2012–2017)** und werden unverändert auf Embargo und Test angewendet.

Warum nur Training? Im Test soll das Modell so arbeiten, als läge die Zukunft noch vor uns. Würden wir Grenzen oder Mediane mit Testdaten berechnen, flösse Wissen aus dem Testzeitraum ins Modell.

Ergebnis: `data/processed/model_data_palepu.csv`. **Keine Modelle.**

## 1. Variablen bauen (identisch zu Notebook 03)

Derselbe Code wie in 03, kompakt zusammengefasst; am Ende prüfen wir, dass dieselben Zahlen herauskommen.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data"
Y = "y_pe_main_h3"

panel = pd.read_csv(DATA / "processed" / "firm_year_panel.csv", dtype={"cik": str})
ratios = pd.read_csv(DATA / "processed" / "firm_year_ratios.csv", dtype={"cik": str})
fund = pd.read_csv(DATA / "processed" / "firm_year_fundamentals.csv", dtype={"cik": str})

ratio_cols = ["sales_growth_1y", "leverage", "log_assets", "float_to_book", "float_to_earnings"]
fund_cols = ["net_income", "equity", "cash", "liabilities_current", "total_assets"]
df = (panel.merge(ratios[["accession"] + ratio_cols], on="accession", validate="one_to_one")
           .merge(fund[["accession"] + fund_cols], on="accession", validate="one_to_one"))

# Stichprobenfilter (Notebook 02)
s = df[df["h3_observed"]]
s = s[~s["fin_sic_60_64"].astype(bool)]
s = s[s["completion_status"] != "uncertain"]
s = s[~pd.to_numeric(s["sic"], errors="coerce").isin([6770])]
s = s[(s["entity_public_float"] >= 50e6) | s["first_listing_year_exempt"].astype(bool)]
sample = s.copy()

# roe und liquidity (Notebook 03, Abschnitte 1-2)
sample["roe"] = sample["net_income"] / sample["equity"].where(sample["equity"] > 0)
sample["liquidity"] = (sample["cash"] - sample["liabilities_current"]) / sample["total_assets"].where(sample["total_assets"] > 0)

# grdummy (Abschnitt 3): hoch/niedrig relativ zum Median des Geschäftsjahres
def high_vs_year_median(col):
    high = sample[col] > sample.groupby("fiscal_year")[col].transform("median")
    return high.where(sample[col].notna())

g, l, v = (high_vs_year_median(c) for c in ["sales_growth_1y", "liquidity", "leverage"])
combo = ((g == False) & (l == True) & (v == False)) | ((g == True) & (l == False) & (v == True))
sample["grdummy"] = combo.astype(float).where(g.notna() & l.notna() & v.notna())

# idummy (Abschnitt 4): pe_main-Event (angekündigt laut first_event_date) in derselben 2-stelligen SIC im Vorjahr
events = pd.read_csv(DATA / "interim" / "combined_events.csv", dtype={"cik": str})
events = events[events["sponsor_like_ptp"].astype(str) == "True"].copy()
events["event_date"] = pd.to_datetime(events["first_event_date"])
events["event_year"] = events["event_date"].dt.year
filings = pd.read_csv(DATA / "interim" / "stage_11_raw_quarters.csv", dtype={"cik": str},
                      usecols=["cik", "sic", "filed_date"]).dropna(subset=["sic"])
filings["filed_date"] = pd.to_datetime(filings["filed_date"])
f = filings.merge(events[["cik", "event_date"]], on="cik")
f["before"] = f["filed_date"] <= f["event_date"]
f["gap"] = (f["filed_date"] - f["event_date"]).abs()
f = pd.concat([f[f["before"]].sort_values("filed_date", ascending=False).groupby("cik").head(1),
               f[~f["before"]].sort_values("gap").groupby("cik").head(1)]).drop_duplicates("cik")
events = events.merge(f[["cik", "sic"]], on="cik", how="left")
events["sic2"] = events["sic"] // 100
active = events.dropna(subset=["sic2"]).groupby(["sic2", "event_year"]).size().rename("n_events").reset_index()

sample["sic2"] = pd.to_numeric(sample["sic"], errors="coerce") // 100
sample["lookup_year"] = sample["fiscal_year"] - 1
sample = sample.merge(active.rename(columns={"event_year": "lookup_year"}), on=["sic2", "lookup_year"], how="left")
sample["idummy"] = (sample["n_events"].fillna(0) > 0).astype(float).where(sample["sic2"].notna())

# Kontrolle gegen Notebook 03
assert len(sample) == 33_428 and sample[Y].sum() == 282
print(f"grdummy = 1: {sample['grdummy'].mean():.1%} (03: 26.6%) | idummy = 1: {sample['idummy'].mean():.1%} (03: 30.1%)")

grdummy = 1: 26.6% (03: 26.6%) | idummy = 1: 30.1% (03: 30.1%)


## 2. Train / Embargo / Test markieren

In [2]:
def split_of(year):
    if year <= 2017:
        return "train"
    if year <= 2019:
        return "embargo"
    return "test"

sample["split"] = sample["fiscal_year"].map(split_of)
train = sample["split"] == "train"
sample.groupby("split", sort=False).agg(Zeilen=("accession", "size"), Positive=(Y, "sum"))

,Zeilen,Positive
split,,
train,18058,167
embargo,5774,45
test,9596,70


## 3. Grenzen und Mediane aus dem Training

Für jede stetige Variable: 1. und 99. Perzentil im Training (die Kappungsgrenzen) und der Median der **gekappten** Trainingswerte (der Füllwert). `grdummy` und `idummy` sind 0/1-Variablen und werden nicht gekappt.

In [3]:
continuous = ["roe", "sales_growth_1y", "liquidity", "leverage", "log_assets", "float_to_book", "float_to_earnings"]

# Nur Trainingszeilen verwenden
lo = sample.loc[train, continuous].quantile(0.01)
hi = sample.loc[train, continuous].quantile(0.99)
median = sample.loc[train, continuous].clip(lo, hi, axis=1).median()
median["idummy"] = sample.loc[train, "idummy"].median()       # Füllwert für die wenigen fehlenden idummy-Werte

params = pd.DataFrame({"untere Grenze (P1)": lo, "obere Grenze (P99)": hi, "Median (Füllwert)": median})
params.round(4)

,untere Grenze (P1),obere Grenze (P99),Median (Füllwert)
float_to_book,-99.6867,195.2249,1.9212
float_to_earnings,-934.5049,449.6903,11.4569
idummy,NaN,NaN,0.0000
leverage,0.0000,2.6510,0.2079
liquidity,-8.7509,0.8327,-0.0763
log_assets,9.5821,25.1876,20.6863
roe,-5.9542,1.2500,0.0692
sales_growth_1y,-0.7349,4.8617,0.0546


## 4. Anwenden auf alle Zeilen

Reihenfolge: zuerst merken, was fehlt (Indikator), dann kappen, dann auffüllen. Welche Indikatoren behalten werden, entscheidet danach die Trainingsregel (>2 % fehlend und ≥5 Positive in der Lücke).

In [4]:
out = sample.copy()
counts = []                                            # Protokoll: wie viel wurde pro Variable verändert?

for v in continuous + ["idummy"]:
    missing = out[v].isna()
    if v in continuous:
        too_low, too_high = out[v] < lo[v], out[v] > hi[v]
        out[v] = out[v].clip(lo[v], hi[v])              # kappen mit den Trainingsgrenzen
    else:
        too_low = too_high = pd.Series(False, index=out.index)
    if missing.any():
        out[f"{v}_missing"] = missing.astype(int)       # 1 = Wert war ursprünglich leer
    out[v] = out[v].fillna(median[v])                   # auffüllen mit dem Trainingsmedian
    for part in ["train", "embargo", "test"]:
        m = out["split"] == part
        counts.append({"Variable": v, "Abschnitt": part, "imputiert": int(missing[m].sum()),
                       "unten gekappt": int(too_low[m].sum()), "oben gekappt": int(too_high[m].sum()),
                       "Zeilen": int(m.sum())})

# grdummy: fehlende Zutat -> 0 plus Indikator (eigene Regel, kein Median)
gr_missing = out["grdummy"].isna()
out["grdummy_missing"] = gr_missing.astype(int)
out["grdummy"] = out["grdummy"].fillna(0)
for part in ["train", "embargo", "test"]:
    m = out["split"] == part
    counts.append({"Variable": "grdummy", "Abschnitt": part, "imputiert": int(gr_missing[m].sum()),
                   "unten gekappt": 0, "oben gekappt": 0, "Zeilen": int(m.sum())})

counts = pd.DataFrame(counts)
assert out[continuous + ["grdummy", "idummy"]].notna().all().all()   # keine Lücken mehr

# Indikator-Regel, nur auf dem Training geprüft: >2 % fehlend UND >= 5 Positive in der Lücke
rule = []
for ind in sorted(c for c in out.columns if c.endswith("_missing")):
    share = out.loc[train, ind].mean()
    pos_in_gap = int(out.loc[train & (out[ind] == 1), Y].sum())
    keep = share > 0.02 and pos_in_gap >= 5
    rule.append({"Indikator": ind, "fehlend im Training": f"{share:.1%}",
                 "Positive in der Lücke": pos_in_gap, "behalten": "ja" if keep else "nein"})
    if not keep:
        out = out.drop(columns=ind)          # Werte bleiben mit dem Median gefüllt, nur der Indikator fällt weg
pd.DataFrame(rule).set_index("Indikator")

,fehlend im Training,Positive in der Lücke,behalten
Indikator,,,
float_to_book_missing,8.7%,20,ja
float_to_earnings_missing,6.1%,7,ja
grdummy_missing,39.1%,50,ja
idummy_missing,0.0%,0,nein
leverage_missing,2.4%,4,nein
liquidity_missing,12.2%,18,ja
log_assets_missing,0.9%,3,nein
roe_missing,12.2%,29,ja
sales_growth_1y_missing,31.5%,37,ja


In [5]:
print("Verbleibende Indikatoren:", sorted(c for c in out.columns if c.endswith("_missing")))

Verbleibende Indikatoren: ['float_to_book_missing', 'float_to_earnings_missing', 'grdummy_missing', 'liquidity_missing', 'roe_missing', 'sales_growth_1y_missing']


## 5. Wie viel wurde verändert? (Train vs. Test)

In [6]:
order = ["roe", "sales_growth_1y", "liquidity", "leverage", "grdummy", "idummy",
         "log_assets", "float_to_book", "float_to_earnings"]
c = counts[counts["Abschnitt"].isin(["train", "test"])].copy()
c["gekappt"] = c["unten gekappt"] + c["oben gekappt"]
c["imputiert %"] = (c["imputiert"] / c["Zeilen"]).map("{:.1%}".format)
c["gekappt %"] = (c["gekappt"] / c["Zeilen"]).map("{:.1%}".format)
table = c.pivot(index="Variable", columns="Abschnitt",
                values=["imputiert", "imputiert %", "gekappt", "gekappt %"]).loc[order]
table = table.reindex(columns=["train", "test"], level=1)
table

imputiert       imputiert %        gekappt      gekappt %  \
Abschnitt             train  test       train   test   train test     train   
Variable                                                                      
roe                    2204  1102       12.2%  11.5%     318  216      1.8%   
sales_growth_1y        5689  1941       31.5%  20.2%     248  260      1.4%   
liquidity              2201   937       12.2%   9.8%     318  202      1.8%   
leverage                428   186        2.4%   1.9%     177   34      1.0%   
grdummy                7058  2685       39.1%  28.0%       0    0      0.0%   
idummy                    1    35        0.0%   0.4%       0    0      0.0%   
log_assets              167    44        0.9%   0.5%     358  165      2.0%   
float_to_book          1569   513        8.7%   5.3%     330  272      1.8%   
float_to_earnings      1105   400        6.1%   4.2%     340  274      1.9%   

                         
Abschnitt          test  
Variable                 
roe                2.3%  
sales_growth_1y    2.7%  
liquidity          2.1%  
leverage           0.4%  
grdummy            0.0%  
idummy             0.0%  
log_assets         1.7%  
float_to_book      2.8%  
float_to_earnings  2.9%

In [7]:
# Unten und oben getrennt, für den Test: wo liegen die Testwerte außerhalb der Trainingsgrenzen?
counts[counts["Abschnitt"] == "test"].set_index("Variable").loc[
    [v for v in order if v in continuous], ["unten gekappt", "oben gekappt", "Zeilen"]]

,unten gekappt,oben gekappt,Zeilen
Variable,,,
roe,109,107,9596
sales_growth_1y,136,124,9596
liquidity,30,172,9596
leverage,0,34,9596
log_assets,26,139,9596
float_to_book,98,174,9596
float_to_earnings,135,139,9596


**Ergebnis:** Gekappt werden pro Variable etwa 1–3 % der Werte – im Training per Definition rund 2 %, im Test ähnlich viel (am meisten bei `float_to_book` und `float_to_earnings` mit knapp 3 %), d. h. die Testjahre liegen kaum außerhalb dessen, was das Training kennt. Aufgefüllt wird vor allem `sales_growth_1y` und damit `grdummy`: im Training 31,5 % bzw. 39,1 %, im Test nur 20,2 % bzw. 28,0 % – der Unterschied kommt fast ganz daher, dass 2012 (nur im Training) kein Vorjahr hat. Die Grenzen sind teils sehr weit (z. B. `log_assets` ab 9,6 ≈ 15.000 USD Bilanzsumme, `liquidity` bis −8,8): Das sind Firmen mit großem Float, aber fast leerer Bilanz, die auch nach dem Kappen extreme Werte behalten.

## 6. Speichern

Gespeichert werden Schlüssel, Abschnitt, beide Labels, die aufbereiteten Variablen und die Fehlend-Indikatoren. Die Rohwerte stehen weiterhin in den Pipeline-Dateien; die Complete-Case-Variante lässt sich später über die Indikatoren bilden (alle `_missing = 0`).

In [8]:
keys = ["cik", "company_name", "fiscal_year", "accession", "sic", "sic2", "split"]
labels = ["y_pe_main_h3", "y_pe_strict_h3"]
indicators = sorted(c for c in out.columns if c.endswith("_missing"))
model_data = out[keys + labels + order + indicators].sort_values(["cik", "fiscal_year"])

path = DATA / "processed" / "model_data_palepu.csv"
model_data.to_csv(path, index=False)
print(f"Gespeichert: {path} - {model_data.shape[0]:,} Zeilen, {model_data.shape[1]} Spalten")
print(model_data["split"].value_counts().to_string())
model_data.head(3)

Gespeichert: ../data/processed/model_data_palepu.csv - 33,428 Zeilen, 24 Spalten
split
train      18058
test        9596
embargo     5774


,cik,company_name,fiscal_year,accession,sic,sic2,split,y_pe_main_h3,y_pe_strict_h3,roe,...,idummy,log_assets,float_to_book,float_to_earnings,float_to_book_missing,float_to_earnings_missing,grdummy_missing,liquidity_missing,roe_missing,sales_growth_1y_missing
0,0000001750,AAR CORP,2012,0001047469-12-007300,3720.0,37.0,train,0,0,0.078324,...,0.0,21.509745,0.809043,10.329398,0,0,1,0,0,1
1,0000001750,AAR CORP,2013,0001047469-13-007797,3720.0,37.0,train,0,0,0.059874,...,0.0,21.482622,0.629326,10.510887,0,0,0,0,0,0
2,0000001750,AAR CORP,2014,0001047469-14-006243,3720.0,37.0,train,0,0,0.072936,...,0.0,21.511496,1.169585,16.035665,0,0,0,0,0,0


## Kernergebnisse

1. Der Modelldatensatz `model_data_palepu.csv` hat 33.428 Zeilen (18.058 Train, 5.774 Embargo, 9.596 Test) mit 9 Palepu-Variablen ohne Lücken; alle Grenzen und Füllwerte stammen nur aus dem Training.
2. Kappen betrifft nur 1–3 % der Werte, Auffüllen dagegen viel – am meisten beim Umsatzwachstum und `grdummy` (im Training rund ein Drittel), weil 2012 kein Vorjahr hat.
3. Nach der Trainingsregel bleiben 6 Fehlend-Indikatoren (roe, sales_growth_1y, liquidity, float_to_book, float_to_earnings, grdummy); `idummy_missing`, `log_assets_missing` und `leverage_missing` fallen weg, weil zu wenige Positive in ihrer Lücke liegen.